# 二つ玉低気圧の見直し（13件）

`japan_sea_low` と `nankigan_low` の両方が付いていて `futatsudama_low` が
付いていない13件を、**以前の答えを見ずに**判定し直します。

二つ玉低気圧は `japan_sea_low` と `nankigan_low` を**置き換える**運用です
(106件のうち56件がこのラベル単独、両方を伴う例は0件)。この13件はその運用から
外れているため、本当に二つ玉なのかを天気図で確かめます。

なお、二つ玉に片方だけを伴っていた9件は判定の要らない規約の問題だったので、
`scripts/apply_label_rule.py` で既に直してあります。

- 結果は `data/review_futatsudama.csv` に書かれます。**元のラベルCSVは変更しません。**
- 途中で閉じても、続きから再開できます。
- 迷ったら「わからない」を押してください。無理に決めると測定の意味が薄れます。
- **元のラベルは伏せられます。** 「日本海低気圧と南岸低気圧が付いている」と見えて
  いると、それに引きずられます。

In [ ]:
import sys
from pathlib import Path

# リポジトリの場所を自動で探す(ノートブックをどこから開いても動くように)
here = Path.cwd()
repo = next((p for p in [here, *here.parents] if (p / "src" / "labels.py").exists()), None)
if repo is None:
    raise SystemExit("リポジトリのルートが見つかりません")
sys.path.insert(0, str(repo))

# 天気図の画像がある場所。環境に合わせて書き換えてください。
IMAGES_DIR = repo.parent / "weather-pattern-classification-data" / "processed"
LABELS_CSV = repo / "data" / "labels_v2.csv"
OUT_CSV = repo / "data" / "review_futatsudama.csv"

print("画像:", IMAGES_DIR, "(あり)" if IMAGES_DIR.exists() else "(見つかりません)")
print("ラベル:", LABELS_CSV, "(あり)" if LABELS_CSV.exists() else "(見つかりません)")

## 判定基準（始める前に記入してください）

書き留めてから始めると、途中で基準が揺れにくくなります。オホーツク海高気圧の
見直しでは、基準を言語化していなかったために κ = 0.464（同じ人が同じ天気図を
見て、半分程度しか一致しない）という結果でした。

> 二つ玉低気圧を「あり」とするのは、
>
> - 日本海側と南岸(太平洋側)に低気圧の中心がそれぞれ1つずつあり、
> - かつ ……（ここに条件を書く。例: 2つが同程度の勢力で、同時に東進していると読めるとき。
>   片方が明らかに衰弱していて主従がはっきりしている場合は、強いほうの単独とする）
>
> 迷う場合の扱い: ……

決めた基準は `src/labels.py` に書き足してください。CSVだけ直すと、次に
ラベルを付けるときに同じ揺れが再発します。

In [ ]:
# 対象の13件を取り出す。候補CSVがなければ先に作る。
#   python -m scripts.label_stats --labels data/labels_v2.csv --label futatsudama_low \
#       --implied-by japan_sea_low nankigan_low --out-csv data/review_futatsudama_candidates.csv
import pandas as pd

candidates = pd.read_csv(repo / "data" / "review_futatsudama_candidates.csv")
targets = candidates.loc[candidates["kind"] == "needs_judgement", "filename"].tolist()
print(f"判定するのは{len(targets)}件")
for name in targets:
    print(" ", name)

In [ ]:
from scripts.label_tool import run_binary_review_session

# sample=None で、filenames に渡した13件すべてを順に見る
run_binary_review_session(
    images_dir=IMAGES_DIR,
    labels_csv=LABELS_CSV,
    out_csv=OUT_CSV,
    label="futatsudama_low",
    filenames=targets,
    sample=None,
)

## 判定が終わったら

`data/review_futatsudama.csv` に結果が入ります。**まだ元のラベルは変わっていません。**

`yes` と答えたものについて、`futatsudama_low` を付け、`japan_sea_low` と
`nankigan_low` を外す(置き換えの運用に合わせる)ことになります。
反映する前に、何件が `yes` だったかを確認してください。

In [ ]:
review = pd.read_csv(OUT_CSV)
print(review["answer"].value_counts().to_string())
print()
print(review.to_string(index=False))